In [26]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt

import config
from src.io import load_model, save_figure, save_table, logger
from src.models import xgb_safe_frame, xgb_feature_name_map
from src.explainability import (
    compute_feature_importance,
    get_biomarker_ranking,
    run_explainability,
)
from src.visualization import setup_style, plot_feature_importance
# 👇 اضافه کردن ایمپورت مهندسی ویژگی
from src.feature_selection import create_engineered_features 

setup_style()

In [27]:
# Load the best model (from Model Training notebook)
model = load_model("best_model_xgboost.joblib")

# Load selected features list
selected_df = pd.read_csv(config.TABLES_DIR / "selected_features.csv")
selected_features = selected_df["feature"].tolist()

# Load RAW test data first
X_test_raw = pd.read_csv(config.PROCESSED_DIR / "X_test_preprocessed.csv")
y_test = pd.read_csv(config.PROCESSED_DIR / "y_test.csv").iloc[:, 0]

# ============================================================
# 👇 FIX: Apply Feature Engineering to Test Data 👇
# ============================================================
print("Applying feature engineering to Test set to match training...")
X_test_eng, _ = create_engineered_features(X_test_raw)

# Now select the specific features the model was trained on
# This ensures columns like 'Gleason_Total' exist before selection
try:
    X_test_selected = X_test_eng[selected_features]
except KeyError as e:
    print(f"❌ Error: Missing features in engineered test data: {e}")
    raise e

print(f"Model loaded: {type(model).__name__}")
print(f"Selected features: {len(selected_features)}")
print(f"Test samples: {len(X_test_selected)}")

2026-08-14 21:39:24 | INFO     | prostate_bcr | Loading model from D:\Prostate_BCR\core\outputs\models\best_model_xgboost.joblib


2026-08-14 21:39:25 | INFO     | prostate_bcr | Engineered: Gleason_Total, High_Risk_Gleason
2026-08-14 21:39:25 | INFO     | prostate_bcr | Engineered: Margin_x_LymphNode
2026-08-14 21:39:25 | INFO     | prostate_bcr | Engineered: T_Stage_Risk
2026-08-14 21:39:25 | INFO     | prostate_bcr | Engineered: PSA_Pathway_Score (from 7 genes)
2026-08-14 21:39:25 | INFO     | prostate_bcr | Engineered: AR_Signaling_Score (from 8 genes)
2026-08-14 21:39:25 | INFO     | prostate_bcr | Engineered: Proliferation_Score (from 7 genes)


Applying feature engineering to Test set to match training...
Model loaded: XGBClassifier
Selected features: 30
Test samples: 129


In [28]:
# Map XGBoost-safe names back to original feature names
name_map = xgb_feature_name_map(selected_features)

importance_df = compute_feature_importance(
    model,
    selected_features,
    top_k=20,
)

print("Top 20 Features by Importance:")
print(importance_df.to_string(index=False))

ValueError: All arrays must be of the same length

In [ ]:
fig = plot_feature_importance(
    importance_df,
    top_k=20,
    title="Top 20 Features — XGBoost Importance",
    filename="feature_importance_top20.png",
)
plt.show()

2026-08-14 21:38:14 | INFO     | prostate_bcr | Saved figure → D:\Prostate_BCR\core\outputs\figures\feature_importance_top20.png
2026-08-14 21:38:14 | INFO     | prostate_bcr | Feature importance plot generated (20 features)


In [ ]:
from src.preprocessing import identify_column_groups

# Identify clinical vs gene columns from the SELECTED features
clinical_cols, gene_cols = identify_column_groups(X_test_selected)

# Rank gene biomarkers
gene_importance = importance_df[importance_df["feature"].isin(gene_cols)].copy()
gene_ranking = get_biomarker_ranking(gene_importance, feature_type="gene")

# Rank clinical biomarkers
clinical_importance = importance_df[importance_df["feature"].isin(clinical_cols)].copy()
clinical_ranking = get_biomarker_ranking(clinical_importance, feature_type="clinical")

print("Top Gene Biomarkers:")
print(gene_ranking.head(10).to_string(index=False))

print("\nTop Clinical Biomarkers:")
print(clinical_ranking.head(10).to_string(index=False))

2026-08-14 21:38:27 | INFO     | prostate_bcr | Column groups: 1 clinical, 29 gene
2026-08-14 21:38:27 | INFO     | prostate_bcr | Biomarker ranking: 29 gene features
2026-08-14 21:38:27 | INFO     | prostate_bcr | Biomarker ranking: 1 clinical features


Top Gene Biomarkers:
 rank feature  importance feature_type
    1  DYNLT1    0.058162         gene
    2   PRAG1    0.058010         gene
    3  PTGER1    0.036174         gene
    4   AS3MT    0.031097         gene
    5   FGF20    0.027936         gene
    6   AIFM3    0.027176         gene
    7   CNTRL    0.026806         gene
    8   SEPT1    0.026609         gene
    9    SHC2    0.025770         gene
   10    PIM2    0.025727         gene

Top Clinical Biomarkers:
 rank                                              feature  importance feature_type
    1 Primary Lymph Node Presentation Assessment Ind-3_YES    0.074959     clinical


In [ ]:
save_table(gene_ranking, "biomarker_ranking_genes.csv", index=False)
save_table(clinical_ranking, "biomarker_ranking_clinical.csv", index=False)
save_table(importance_df, "feature_importance_full.csv", index=False)

print("Biomarker rankings saved to outputs/tables/")

2026-08-14 21:38:41 | INFO     | prostate_bcr | Saved 29 rows → D:\Prostate_BCR\core\outputs\tables\biomarker_ranking_genes.csv
2026-08-14 21:38:41 | INFO     | prostate_bcr | Saved 1 rows → D:\Prostate_BCR\core\outputs\tables\biomarker_ranking_clinical.csv
2026-08-14 21:38:41 | INFO     | prostate_bcr | Saved 37 rows → D:\Prostate_BCR\core\outputs\tables\feature_importance_full.csv


Biomarker rankings saved to outputs/tables/


In [ ]:
# Run full explainability pipeline with SHAP
# Note: We pass X_test_selected which now includes engineered features
results = run_explainability(
    model=model,
    X_test=X_test_selected, 
    feature_names=selected_features,
    top_k=20,
    run_shap=True,
    sample_index=0,
)

if "shap_error" in results:
    print(f"SHAP analysis skipped: {results['shap_error']}")
else:
    print("SHAP analysis completed successfully.")

ValueError: All arrays must be of the same length

In [ ]:
if "summary_plot" in results:
    fig = results["summary_plot"]
    save_figure(fig, "shap_summary_plot.png")
    plt.show()
else:
    print("SHAP summary plot not available.")

In [ ]:
if "waterfall_plot" in results:
    fig = results["waterfall_plot"]
    save_figure(fig, "shap_waterfall_sample0.png")
    plt.show()
else:
    print("SHAP waterfall plot not available.")

In [ ]:
if "shap_values" in results:
    from src.explainability import plot_shap_dependence

    top_feature = importance_df.iloc[0]["feature"]
    fig = plot_shap_dependence(
        results["shap_values"],
        X_test_selected, # Use the engineered dataframe here too
        feature_name=top_feature,
        figsize=(8, 6),
    )
    save_figure(fig, f"shap_dependence_{top_feature[:20]}.png")
    plt.show()
    print(f"Dependence plot generated for: {top_feature}")
else:
    print("SHAP values not available.")

In [ ]:
# Save the final feature list (including engineered ones) just in case
pd.DataFrame({"feature": selected_features}).to_csv(
    config.TABLES_DIR / "selected_features_final.csv", index=False
)
print("Saved selected_features_final.csv")